# Width dependence of dropout scheduling

The mean-field argument assumes wide layers, so varying width tests how far its scheduling prediction carries into finite networks. We vary the hidden width from 64 to 1024 while holding depth fixed and the mean dropout probability at $\bar p=0.1$, giving each width its own no-dropout control. The ReLU MLP uses weight variance $\sigma_w^2=2$ and zero biases.

For the saved results, run **Setup**, then **Saved results** and **Figures**. Running **Training** repeats the sweep with ten seeds per configuration.


## Setup


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
if not (ROOT / "utils").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "utils").is_dir():
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RESULTS_DIR = ROOT / "results"
RUN_DIR = ROOT / "runs" / "mlp_width_sweep"
RUN_DIR.mkdir(parents=True, exist_ok=True)
RESULT_FILE = "mlp_width_sweep.npz"

import torch
import numpy as np
import matplotlib.pyplot as plt
import ast
import time
from benchmarks.original.relu import get_dropout_schedule, CriticalReLUNet, train_epoch, evaluate
from functools import partial
from benchmarks.original import relu
from utils.cifar import load_cifar10 as load_cifar10_data
from utils.results import save_npz_result, load_npz_result
from scipy.stats import sem as standard_error

from utils.experiments import run_simulations as run_mlp_simulations
from utils.sweep_plots import (
    DISPLAY_NAMES,
    sweep_style,
    sweep_accuracy,
    sweep_advantage,
    sweep_table,
    training_panels,
    all_training_curves,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

if device == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    print(f"GPU: {torch.cuda.get_device_name(0)}")

USE_AMP = device == "cuda"
AMP_DTYPE = torch.bfloat16 if (USE_AMP and torch.cuda.is_bf16_supported()) else torch.float16

In [ ]:
# Configuration

N_SIMULATIONS = 10

# Fixed mean dropout field
H_BAR = 0.1

# Width sweep: 64, 128, 256, 512, 1024
WIDTH_VALUES = [64, 128, 256, 512, 1024]

# Schedules to compare
SCHEDULES = ["none", "constant", "reverse_step", "big_step"]

# Network geometry (depth is fixed across all widths)
SIGMA_B_SQ = 0.00
SIGMA_W_SQ = 2
DEPTH_MULTIPLIER = 2

# Depth is fixed using the h_bar = 0.1 correlation length.
kappa = 2 * np.sqrt(2) / (3 * np.pi)
alpha = (3 / 2) * kappa ** (2 / 3)
xi_ref = 1 / (alpha * H_BAR ** (1 / 3))
DEPTH = int(DEPTH_MULTIPLIER * xi_ref)

# Training
EPOCHS = 75
LEARNING_RATE = 1e-4
BATCH_SIZE = 75
USE_LR_SCHEDULE = True
LR_MIN = 1e-7

# Dataset
USE_FULL_DATASET = False
TRAIN_SIZE = None if USE_FULL_DATASET else 5000
TEST_SIZE = None if USE_FULL_DATASET else 5000

# Plotting - selected width for single-width training curves figure
WIDTH_PLOT = 256  # must be in WIDTH_VALUES

# Display / colour / marker maps (shared by all figures)
print(f"Fixed h_bar: {H_BAR}")
print(f"Fixed depth: L = {DEPTH}  (from {DEPTH_MULTIPLIER} x xi at h_bar = {H_BAR})")
print(f"Width values: {WIDTH_VALUES}")
print(f"L/N ratios:   {[round(DEPTH / w, 3) for w in WIDTH_VALUES]}")
print(f"Schedules: {SCHEDULES}")
print(f"Simulations per config: {N_SIMULATIONS}")
total_runs = len(WIDTH_VALUES) * len(SCHEDULES)
print(f"Total training runs: {total_runs * N_SIMULATIONS}")

## Correlation length

The code calls the mean dropout probability `H_BAR` and the layerwise probabilities `h_layers`. These are proxies for the theoretical field $h=1-F(1)$ in the near-critical estimate $\xi_{\mathrm{eff}}^{-1}\propto\langle p_\ell^{1/3}\rangle$. That estimate is the same at every width for a given schedule, while the ratio $L/N$ changes, making finite-width effects visible in the training comparison.


In [ ]:
def compute_effective_xi(h_layers):
    """Compute effective correlation length for ReLU (kinked) networks."""
    L = len(h_layers)
    damage = sum(h ** (1 / 3) for h in h_layers if h > 0) / L
    xi_inv = alpha * damage
    return 1 / xi_inv if xi_inv > 0 else float("inf")

In [ ]:
# Theoretical predictions for every (width, schedule) pair
# (xi_eff depends only on the schedule, but we index by width for consistency)

theory = {}

print(
    f"{'Width':<8} {'Schedule':<16} {'h_mean':>8} {'h_max':>8} {'xi_eff':>8} {'L/xi':>6} {'L/N':>6}"
)
print("-" * 70)

h_max = 2 * H_BAR
for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, h_max)
        xi = compute_effective_xi(h_layers)
        theory[(width, sched)] = {"h_layers": h_layers, "xi_eff": xi}

        xi_str = f"{xi:.2f}" if xi < 1e6 else "inf"
        ratio = DEPTH / xi if xi < 1e6 else 0
        print(
            f"{width:<8d} {sched:<16} {np.mean(h_layers):>8.4f} "
            f"{max(h_layers):>8.4f} {xi_str:>8} {ratio:>6.2f} {DEPTH / width:>6.3f}"
        )

## Training

This section downloads CIFAR-10 and trains the models with the configuration above. New results and figures are written to `runs/mlp_width_sweep/`.


In [ ]:
relu.USE_AMP, relu.AMP_DTYPE = USE_AMP, AMP_DTYPE
load_cifar10 = partial(load_cifar10_data, root=ROOT, device=device)
# Training utilities

In [ ]:
# Load data

print("Loading CIFAR-10...")
train_data, test_data = load_cifar10(TRAIN_SIZE, TEST_SIZE)
print(f"Train shape: {train_data[0].shape}")

In [ ]:
# Run the full width sweep


def run_simulations(h_layers, width, n_sims, label):
    return run_mlp_simulations(
        lambda: CriticalReLUNet(3072, width, 10, h_layers, SIGMA_W_SQ, SIGMA_B_SQ).to(device),
        train_data,
        test_data,
        n_sims,
        label,
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        learning_rate=LEARNING_RATE,
        lr_min=LR_MIN if USE_LR_SCHEDULE else None,
        train=train_epoch,
        evaluate=evaluate,
    )


all_results = {}
t_start = time.time()

h_max = 2 * H_BAR

# Architecture changes with width, so every schedule (including 'none') is
# run separately for each width.
for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        print(f"\n{'=' * 60}")
        print(f"Width = {width} | Schedule: {sched}")
        print(f"{'=' * 60}")

        h_layers = get_dropout_schedule(sched, DEPTH, H_BAR, h_max)
        xi = compute_effective_xi(h_layers)
        if sched == "none":
            print(f"  xi_eff = inf (no dropout), L/N = {DEPTH / width:.3f}")
        else:
            print(f"  h_layers = {[f'{h:.4f}' for h in h_layers]}")
            print(f"  xi_eff = {xi:.2f}, L/xi = {DEPTH / xi:.2f}, L/N = {DEPTH / width:.3f}")

        result = run_simulations(h_layers, width, N_SIMULATIONS, f"w={width} {sched}")
        all_results[(width, sched)] = result

        best = result["test_acc"].max(axis=1)
        print(f"  Best test = {best.mean():.2f}% +/- {standard_error(best):.2f}% (SEM)")

elapsed = time.time() - t_start
print(f"\nTotal time: {elapsed / 60:.1f} min")

In [ ]:
# Save results

save_data = {
    "all_results": {str(k): v for k, v in all_results.items()},
    "theory": {str(k): v for k, v in theory.items()},
    "config": {
        "H_BAR": H_BAR,
        "WIDTH_VALUES": WIDTH_VALUES,
        "SCHEDULES": SCHEDULES,
        "N_SIMULATIONS": N_SIMULATIONS,
        "EPOCHS": EPOCHS,
        "DEPTH": DEPTH,
        "SIGMA_W_SQ": SIGMA_W_SQ,
        "SIGMA_B_SQ": SIGMA_B_SQ,
        "LEARNING_RATE": LEARNING_RATE,
    },
}

save_npz_result(RUN_DIR / RESULT_FILE, save_data)

print(f"Results saved to {RUN_DIR / RESULT_FILE}")

## Saved results

The loader prefers a local rerun when one exists and otherwise reads the checked-in results. To compare against the archived runs regardless of what you have trained locally, set `result_path = RESULTS_DIR / RESULT_FILE`.


In [ ]:
result_path = RUN_DIR / RESULT_FILE
if not result_path.exists():
    result_path = RESULTS_DIR / RESULT_FILE
save_data = load_npz_result(result_path)

# Restore tuple keys: "(256, 'constant')" -> (256, 'constant')
all_results = {ast.literal_eval(k): v for k, v in save_data["all_results"].items()}
theory = {ast.literal_eval(k): v for k, v in save_data["theory"].items()}
config = save_data["config"]

# Restore the saved configuration
H_BAR = config["H_BAR"]
WIDTH_VALUES = config["WIDTH_VALUES"]
SCHEDULES = config["SCHEDULES"]
N_SIMULATIONS = config["N_SIMULATIONS"]
EPOCHS = config["EPOCHS"]
DEPTH = config["DEPTH"]
SIGMA_W_SQ = config["SIGMA_W_SQ"]
SIGMA_B_SQ = config["SIGMA_B_SQ"]
LEARNING_RATE = config["LEARNING_RATE"]

print(f"Loaded {len(all_results)} result entries from {result_path}")
print(f"h_bar: {H_BAR}")
print(f"Width values: {WIDTH_VALUES}")
print(f"Schedules: {SCHEDULES}")

In [ ]:
# Summary table

print(f"{'Width':<8} {'Schedule':<16} {'xi_eff':>8} {'L/xi':>6} {'L/N':>6} {'Best Test Acc':>20}")
print("=" * 75)

for width in WIDTH_VALUES:
    for sched in SCHEDULES:
        xi = theory[(width, sched)]["xi_eff"]
        best = all_results[(width, sched)]["test_acc"].max(axis=1)
        sem = standard_error(best)
        xi_str = f"{xi:.2f}" if xi < 1e6 else "inf"
        ratio = f"{DEPTH / xi:.2f}" if xi < 1e6 else "0.00"
        print(
            f"{width:<8d} {DISPLAY_NAMES[sched]:<16} {xi_str:>8} {ratio:>6} "
            f"{DEPTH / width:>6.3f} {best.mean():>8.2f} +/- {sem:.2f}%"
        )
    print("-" * 75)

## Figures

Accuracy summaries take each seed's highest recorded test accuracy, and the summary and paired-gain error bars show the sample standard error of the mean (SEM, `ddof=1`). The learning curves retain the original mean ± one standard deviation bands (`ddof=0`), which describe the spread across seeds; `scripts/plot_results.py` plots the same saved runs with SEM bands.


In [ ]:
sweep_style()
best = {key: result["test_acc"].max(axis=1) for key, result in all_results.items()}

In [ ]:
# Figure 1: Best test accuracy vs width for each schedule

fig, ax = plt.subplots(figsize=(6, 4))

sweep_accuracy(ax, best, WIDTH_VALUES, SCHEDULES, capthick=1.2)

ax.set_xscale("log", base=2)
ax.set_xticks(WIDTH_VALUES)
ax.set_xticklabels([str(w) for w in WIDTH_VALUES])
ax.set(xlabel=r"Hidden width $N$", ylabel="Best test accuracy (%)")
ax.set_title(rf"Dropout scheduling vs width at $\bar{{h}}={H_BAR}$ (MLP, CIFAR-10)")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(RUN_DIR / "width_sweep_accuracy.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'width_sweep_accuracy.pdf'}")

In [ ]:
# Figure 2: Scheduling advantage (schedule - constant) vs width

fig, ax = plt.subplots(figsize=(6, 3.5))

sweep_advantage(ax, best, WIDTH_VALUES)

ax.set_xscale("log", base=2)
ax.set_xticks(WIDTH_VALUES)
ax.set_xticklabels([str(w) for w in WIDTH_VALUES])
ax.set(xlabel=r"Hidden width $N$", ylabel=r"$\Delta$ best test accuracy (%)")
ax.set_title(rf"Scheduling advantage vs width at $\bar{{h}}={H_BAR}$")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(RUN_DIR / "width_sweep_delta.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'width_sweep_delta.pdf'}")

In [ ]:
# Figure 3: Accuracy vs aspect ratio L/N for each schedule
# (at fixed h_bar, xi_eff is constant in width, so we plot against L/N instead)

fig, ax = plt.subplots(figsize=(6, 4))

ratios = [DEPTH / w for w in WIDTH_VALUES]

sweep_accuracy(ax, best, WIDTH_VALUES, SCHEDULES, x=ratios)

ax.set_xscale("log")
ax.set(xlabel=r"Aspect ratio $L/N$", ylabel="Best test accuracy (%)")
ax.set_title(rf"Accuracy vs aspect ratio at $\bar{{h}}={H_BAR}$")
ax.legend(loc="best")

plt.tight_layout()
plt.savefig(RUN_DIR / "width_sweep_aspect_ratio.pdf")
plt.show()
print(f"Saved: {RUN_DIR / 'width_sweep_aspect_ratio.pdf'}")

In [ ]:
sweep_table(
    best,
    WIDTH_VALUES,
    SCHEDULES,
    caption=rf"Best test accuracy (\%) across network widths $N$ at fixed $\bar{{h}}={H_BAR}$ for different schedules (MLP, CIFAR-10). Errors are SEM over {N_SIMULATIONS} seeds.",
    label="tab:width_sweep",
    heading=r"$N$",
    value_format="",
)

In [ ]:
fig = training_panels(
    all_results,
    WIDTH_PLOT,
    SCHEDULES,
    EPOCHS,
    title=rf"Training curves at $N = {WIDTH_PLOT}$, $\bar{{h}}={H_BAR}$",
)
plt.savefig(RUN_DIR / f"width_{WIDTH_PLOT}_training_curves.pdf", bbox_inches="tight")
plt.savefig(RUN_DIR / f"width_{WIDTH_PLOT}_training_curves.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig = all_training_curves(
    all_results,
    WIDTH_VALUES,
    SCHEDULES,
    EPOCHS,
    row_labels=[rf"$N={width}$" for width in WIDTH_VALUES],
)
plt.savefig(RUN_DIR / "width_sweep_all_curves.pdf", bbox_inches="tight")
plt.savefig(RUN_DIR / "width_sweep_all_curves.png", dpi=300, bbox_inches="tight")
plt.show()